# 01 — Data Compilation and EDA
**SPJIMR ANA526-PPM | TALA Multimodal AI Strategy | Day 1**

## Purpose
Load cleaned Day 1 candidate datasets from `data/interim/day1/`, validate provenance,
and produce EDA outputs that frame the Day 2 modelling and RAG work.

## Evidence Discipline

| Evidence type | Valid sources | Use for |
|--------------|---------------|---------|
| `official` | weartala.com, impact reports, press releases | Claim corpus, RAG |
| `customer_experience` | Trustpilot, Reddit, Google Reviews | Quality, fit, durability, returns |
| `press` | Google News RSS, editorial coverage | Press leads — fetch article body before analysis |
| `community` | Reddit threads | Brand perception, sizing discussions |
| `creator_strategy` | YouTube/Instagram/TikTok creator content | Creator mix, platform strategy |
| `competitor_benchmark` | Competitor profile snapshots | Cross-brand comparison |

> **Rule:** Never use Instagram/TikTok engagement metrics as evidence for product quality or sustainability claims.
> Google News RSS rows are **press leads**, not full articles — `usable_for_analysis=True` means the summary is substantive, not that the article was fetched.

## 0. Setup and Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_colwidth", 120)

INTERIM   = PROJECT_ROOT / "data" / "interim" / "day1"
TABLES    = PROJECT_ROOT / "outputs" / "tables"
FIGURES   = PROJECT_ROOT / "outputs" / "figures"
BRANDS    = ["TALA", "Adanola", "Girlfriend Collective", "Oner Active"]

INTERIM_FILES = {
    "official_claims":      INTERIM / "official_claims_cleaned.csv",
    "customer_reviews":     INTERIM / "customer_reviews_cleaned.csv",
    "press_reddit_sources": INTERIM / "press_reddit_sources_cleaned.csv",
    "creator_posts":        INTERIM / "creator_posts_cleaned.csv",
    "competitor_platforms": INTERIM / "competitor_platforms_cleaned.csv",
}

print("Project root:", PROJECT_ROOT)
print("Interim files found:", sum(p.exists() for p in INTERIM_FILES.values()), "/", len(INTERIM_FILES))

## 1. Load Cleaned Interim Datasets

In [ ]:
dfs = {}
for schema, path in INTERIM_FILES.items():
    if path.exists():
        dfs[schema] = pd.read_csv(path, low_memory=False)
        print(f"  [OK] {schema}: {len(dfs[schema])} rows")
    else:
        print(f"  [MISSING] {path.name} — run scripts/clean_day1_data.py first")

claims      = dfs.get("official_claims",      pd.DataFrame())
reviews     = dfs.get("customer_reviews",     pd.DataFrame())
press       = dfs.get("press_reddit_sources", pd.DataFrame())
creators    = dfs.get("creator_posts",        pd.DataFrame())
competitors = dfs.get("competitor_platforms", pd.DataFrame())

## 2. Row Count and Collection Summary

In [ ]:
summary_path = TABLES / "day1_collection_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary.style.set_caption("Day 1 Collection Summary"))
else:
    print("Run scripts/clean_day1_data.py to generate this table.")
    summary = pd.DataFrame()

# Quick text summary
for _, row in summary.iterrows():
    pct_review = 100 * row["rows_needing_manual_review"] / max(row["cleaned_rows"], 1)
    pct_usable = 100 * row["rows_usable_for_analysis"] / max(row["cleaned_rows"], 1)
    print(f"  {row['file']:<45}  "
          f"cleaned={row['cleaned_rows']:>3}  "
          f"manual_review={pct_review:.0f}%  "
          f"usable={pct_usable:.0f}%")

## 3. Brand / Platform / Evidence Matrix

In [ ]:
matrix_path = TABLES / "day1_brand_platform_matrix.csv"
if matrix_path.exists():
    matrix = pd.read_csv(matrix_path)

    # Brand x evidence_type pivot
    brand_ev = matrix.groupby(["brand", "evidence_type"])["row_count"].sum().unstack(fill_value=0)
    print("Rows by brand x evidence_type:")
    display(brand_ev)

    # Top source platforms
    top_platforms = matrix.groupby("source_platform")["row_count"].sum().sort_values(ascending=False).head(15)
    print("\nTop 15 source platforms:")
    display(top_platforms.to_frame("rows"))
else:
    print("Run scripts/clean_day1_data.py first.")

## 4. Evidence Type Distribution

In [ ]:
label_path = TABLES / "day1_label_distribution.csv"
if label_path.exists():
    labels = pd.read_csv(label_path)
    display(labels.style.set_caption("Provisional Label Distribution"))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: label distribution
    axes[0].barh(labels["provisional_label"][::-1], labels["row_count"][::-1],
                 color=sns.color_palette("Set2", len(labels)))
    axes[0].set_title("Provisional Label Distribution — Day 1 Candidates")
    axes[0].set_xlabel("Row count")
    for i, (_, r) in enumerate(labels[::-1].iterrows()):
        axes[0].text(r["row_count"] + 0.5, i, f'{r["percent"]}%', va="center", fontsize=8)

    # Right: evidence type pie
    if not press.empty:
        combined_ev = pd.concat([
            df["evidence_type"] for df in [claims, reviews, press, creators, competitors]
            if not df.empty
        ])
        ev_counts = combined_ev.value_counts()
        axes[1].pie(ev_counts, labels=ev_counts.index, autopct="%1.0f%%",
                    colors=sns.color_palette("Set2", len(ev_counts)), startangle=140)
        axes[1].set_title("Evidence Type Split — All Schemas")

    plt.tight_layout()
    plt.savefig(FIGURES / "day1_label_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Run scripts/clean_day1_data.py first.")

## 5. Collection Charts

In [ ]:
from IPython.display import Image, display as ipy_display

for fname, caption in [
    ("rows_by_brand.png",          "Rows by Brand"),
    ("rows_by_evidence_type.png",  "Rows by Evidence Type"),
    ("rows_by_source_platform.png","Rows by Source Platform (top 12)"),
]:
    p = FIGURES / fname
    if p.exists():
        print(caption)
        ipy_display(Image(str(p), width=700))
    else:
        print(f"[MISSING] {fname} — run scripts/clean_day1_data.py")

## 6. Missing Provenance Audit

In [ ]:
PROVENANCE = ["source_url", "source_platform", "collection_date", "collected_by", "evidence_type", "brand"]

rows = []
for schema, df in dfs.items():
    for col in PROVENANCE:
        if col not in df.columns:
            pct = None
            status = "MISSING_COL"
        else:
            null_n = int(df[col].isna().sum() + (df[col].astype(str).str.strip() == "").sum())
            pct = round(100 * (1 - null_n / max(len(df), 1)), 1)
            status = "OK" if pct == 100 else f"INCOMPLETE ({null_n} nulls)"
        rows.append({"schema": schema, "field": col, "completeness_%": pct, "status": status})

prov_df = pd.DataFrame(rows)
pivot = prov_df.pivot(index="schema", columns="field", values="completeness_%")
print("Provenance completeness (%) — 100 = fully populated:")
display(pivot)

issues = prov_df[prov_df["status"] != "OK"]
if not issues.empty:
    print("\nIssues requiring attention:")
    display(issues[["schema", "field", "status"]])

## 7. Top Source URLs

In [ ]:
all_frames = []
for schema, df in dfs.items():
    tmp = df[["brand", "source_url", "source_platform", "evidence_type"]].copy()
    tmp["schema"] = schema
    all_frames.append(tmp)

all_rows = pd.concat(all_frames, ignore_index=True)

print("Top 20 source URLs by frequency:")
top_urls = (
    all_rows.groupby(["source_url", "source_platform", "evidence_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)
display(top_urls)

print("\nTop platforms (all schemas combined):")
display(all_rows["source_platform"].value_counts().head(15).to_frame("rows"))

## 8. Manual Review Queue

In [ ]:
TEXT_FIELDS = {
    "official_claims":      "claim_text",
    "customer_reviews":     "review_text",
    "press_reddit_sources": "summary",
    "creator_posts":        "caption",
    "competitor_platforms": "context_text",
}

print("=== Manual Review Queue ===\n")
for schema, df in dfs.items():
    if "needs_manual_review" not in df.columns:
        continue
    queue = df[df["needs_manual_review"] == True]
    if queue.empty:
        continue

    text_col = TEXT_FIELDS.get(schema, "")
    cols = ["brand", "source_url", "review_reason"]
    if text_col and text_col in queue.columns:
        cols.append(text_col)

    print(f"--- {schema} ({len(queue)} rows need review) ---")

    # Breakdown by review reason
    reason_counts = (
        queue["review_reason"]
        .str.split("; ")
        .explode()
        .value_counts()
    )
    for reason, cnt in reason_counts.items():
        print(f"    {reason}: {cnt}")
    print()

# Save a combined manual review queue
queue_frames = []
for schema, df in dfs.items():
    if "needs_manual_review" not in df.columns:
        continue
    q = df[df["needs_manual_review"] == True].copy()
    if q.empty:
        continue
    q["_schema"] = schema
    text_col = TEXT_FIELDS.get(schema, "")
    keep = ["_schema", "brand", "source_url", "source_platform", "evidence_type",
            "review_reason", "usable_for_analysis", "usable_for_rag"]
    if text_col and text_col in q.columns:
        keep.append(text_col)
    queue_frames.append(q[keep])

if queue_frames:
    review_queue = pd.concat(queue_frames, ignore_index=True)
    out = TABLES / "day1_manual_review_queue.csv"
    review_queue.to_csv(out, index=False)
    print(f"Full manual review queue saved: {out} ({len(review_queue)} rows)")

## 9. Customer Reviews — Quick EDA

In [ ]:
if not reviews.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Reviews per brand
    brand_counts = reviews["brand"].value_counts()
    axes[0].bar(brand_counts.index, brand_counts.values,
                color=sns.color_palette("Set2", len(brand_counts)))
    axes[0].set_title("Review Rows by Brand")
    axes[0].set_ylabel("Rows")
    axes[0].tick_params(axis="x", rotation=20)

    # Source platform
    plat_counts = reviews["source_platform"].value_counts().head(8)
    axes[1].barh(plat_counts.index[::-1], plat_counts.values[::-1],
                 color=sns.color_palette("Set2", len(plat_counts)))
    axes[1].set_title("Review Source Platforms")
    axes[1].set_xlabel("Rows")

    # Text length distribution
    reviews["text_len"] = reviews["review_text"].fillna("").str.len()
    axes[2].hist(reviews["text_len"], bins=20,
                 color=sns.color_palette("Set2")[2], edgecolor="white")
    axes[2].axvline(80, color="red", linestyle="--", alpha=0.6, label="Min threshold (80)")
    axes[2].set_title("Review Text Length Distribution")
    axes[2].set_xlabel("Characters")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(FIGURES / "eda_reviews_day1.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Usability breakdown
    print(f"\nReview usability:")
    print(f"  usable_for_analysis:            {reviews['usable_for_analysis'].sum()} / {len(reviews)}")
    print(f"  usable_for_rag:                 {reviews['usable_for_rag'].sum()} / {len(reviews)}")
    print(f"  usable_for_quality_responsibility: {reviews['usable_for_quality_responsibility'].sum()} / {len(reviews)}")
    print(f"\nNote: all rows are DDG search snippets pointing to review pages.")
    print(f"Fetch full page text before using in RAG or sentiment analysis.")
else:
    print("No review data available.")

## 10. Press Sources — Quick EDA

In [ ]:
if not press.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Press rows by brand
    press_by_brand = press.groupby("brand").size().sort_values()
    axes[0].barh(press_by_brand.index, press_by_brand.values,
                 color=sns.color_palette("Set2", len(press_by_brand)))
    axes[0].set_title("Press Lead Rows by Brand")
    axes[0].set_xlabel("Rows")

    # Summary length distribution
    press["summary_len"] = press["summary"].fillna("").str.len()
    axes[1].hist(press["summary_len"], bins=20,
                 color=sns.color_palette("Set2")[1], edgecolor="white")
    axes[1].set_title("Press Summary Length Distribution")
    axes[1].set_xlabel("Characters")

    plt.tight_layout()
    plt.savefig(FIGURES / "eda_press_day1.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nAll {len(press)} press rows are Google News RSS entries (provider=google_news_rss).")
    print(f"They contain article title + summary (~400 chars) — NOT full article body.")
    print(f"usable_for_analysis (summary substantive): {press['usable_for_analysis'].sum()}")
    print(f"Recommended action: fetch article body for top 20 TALA rows before Day 2.")

    print(f"\nTop 5 TALA press summary excerpts:")
    tala_press = press[press["brand"] == "TALA"].head(5)
    for _, row in tala_press.iterrows():
        print(f"  [{row.get('date_published','')[:10]}] {row.get('title','')[:80]}")
else:
    print("No press data available.")

## 11. Official Claims — Quick EDA

In [ ]:
if not claims.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    claims["claim_category"].value_counts().plot(
        kind="barh", ax=axes[0], color=sns.color_palette("Set2")[2])
    axes[0].set_title("Official Claims by Category")
    axes[0].set_xlabel("Count")

    claims["source_type"].value_counts().plot(
        kind="barh", ax=axes[1], color=sns.color_palette("Set2")[3])
    axes[1].set_title("Claims by Source Type")
    axes[1].set_xlabel("Count")

    plt.tight_layout()
    plt.savefig(FIGURES / "eda_claims_day1.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nNote: all {len(claims)} claim rows came from DuckDuckGo search snippets.")
    print(f"They are press/secondary sources referencing TALA claims — NOT verbatim brand copy.")
    print(f"Manual supplement recommended: copy verbatim text from weartala.com/pages/sustainability")
    print(f"\nSample claim texts:")
    for t in claims["claim_text"].head(4):
        print(f"  - {t[:120]}")
else:
    print("No official claims data.")

## 12. Creator Posts — Quick EDA

In [ ]:
if not creators.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    creators.groupby("brand").size().plot(kind="bar", ax=axes[0],
        color=sns.color_palette("Set2", creators["brand"].nunique()))
    axes[0].set_title("Creator Rows by Brand")
    axes[0].tick_params(axis="x", rotation=20)

    creators["platform"].value_counts().plot(kind="bar", ax=axes[1],
        color=sns.color_palette("Set2")[1])
    axes[1].set_title("Creator Rows by Platform")
    axes[1].tick_params(axis="x", rotation=0)

    creators["partnership_type"].value_counts().plot(kind="barh", ax=axes[2],
        color=sns.color_palette("Set2")[2])
    axes[2].set_title("Partnership Type Inferred")
    axes[2].set_xlabel("Rows")

    plt.tight_layout()
    plt.savefig(FIGURES / "eda_creators_day1.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nNote: all {len(creators)} creator rows are DDG search pointers to creator content.")
    print(f"Instagram/TikTok rows marked as candidate evidence only.")
    print(f"usable_for_creator_strategy: {creators['usable_for_creator_strategy'].sum()} / {len(creators)}")
    youtube_rows = creators[creators["platform"] == "youtube"]
    print(f"YouTube rows (most verifiable): {len(youtube_rows)}")
else:
    print("No creator data.")

## 13. Competitor Platforms — Quick EDA

**Evidence type:** `competitor_benchmark` — platform presence pointers for cross-brand comparison.

In [ ]:
if not competitors.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Rows per brand
    competitors.groupby("brand").size().plot(
        kind="bar", ax=axes[0],
        color=sns.color_palette("Set2", competitors["brand"].nunique()))
    axes[0].set_title("Competitor Rows by Brand")
    axes[0].tick_params(axis="x", rotation=20)

    # Rows per source platform
    competitors["source_platform"].value_counts().plot(
        kind="barh", ax=axes[1],
        color=sns.color_palette("Set2")[1])
    axes[1].set_title("Competitor Source Platforms")
    axes[1].set_xlabel("Rows")

    plt.tight_layout()
    plt.savefig(FIGURES / "eda_competitors_day1.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nCompetitor rows: {len(competitors)}")
    print(f"Provisional labels:")
    if "provisional_label" in competitors.columns:
        print(competitors["provisional_label"].value_counts().to_string())
    print(f"\nNote: rows with provider_name='trafilatura' have extracted page text.")
    print(f"DDG pointer rows need manual verification of platform presence data.")
    print(f"usable_for_analysis: {competitors['usable_for_analysis'].sum()} / {len(competitors)}")
else:
    print("No competitor data.")

## 14. Day 2 Evidence Readiness and Roadmap

What is strong enough to carry forward, and what needs human action before Day 2 modelling begins.

In [ ]:
readiness_rows = []
for schema, df in dfs.items():
    if df.empty:
        continue
    n = len(df)
    for col, label in [
        ("usable_for_analysis",           "Feature extraction / EDA"),
        ("usable_for_rag",                "RAG corpus (Day 3)"),
        ("usable_for_creator_strategy",   "Creator strategy analysis"),
        ("usable_for_quality_responsibility", "Quality/sustainability analysis"),
    ]:
        if col in df.columns:
            yes = int(df[col].sum())
            readiness_rows.append({
                "schema": schema,
                "analysis_track": label,
                "usable_rows": yes,
                "total_rows": n,
                "pct_usable": round(100 * yes / n, 1),
            })

rdf = pd.DataFrame(readiness_rows)
if not rdf.empty:
    pivot_ready = rdf.pivot_table(
        index="schema", columns="analysis_track",
        values="pct_usable", aggfunc="first"
    ).fillna(0).round(1)
    print("Evidence readiness by schema and analysis track (% of cleaned rows):")
    display(pivot_ready.style.background_gradient(cmap="YlGn", axis=None, vmin=0, vmax=100))

print("\n=== Day 2 Action List ===")
print()
print("READY FOR DAY 2 (no action needed):")
print("  - press_reddit_sources: 127 press leads with summaries -> run TF-IDF topic modelling")
print("  - official_claims: 24 DDG-sourced claim snippets -> seed RAG claims corpus")
print("  - creator_posts: 36 DDG pointers -> platform/partnership type analysis")
print()
print("NEEDS HUMAN ACTION BEFORE DAY 2:")
print("  - customer_reviews: all 56 rows are DDG snippets, not extracted review text")
print("    Action: open top 10 Trustpilot URLs manually; paste review text into cleaned CSV")
print("  - official_claims: not verbatim brand copy")
print("    Action: copy text from weartala.com/pages/sustainability + /about + /responsibility")
print("  - press leads: article body not fetched")
print("    Action: run scripts/fetch_article_bodies.py (Day 2) on top TALA press rows")
print()
print("NEEDS APIFY OR YOUTUBE API TO IMPROVE:")
print("  - customer_reviews: Trustpilot full reviews (Apify monthly limit hit; ~$3 to buy more)")
print("  - creator_posts: YouTube API key would add video metadata (currently DDG only)")

## 15. Notebook Summary

In [ ]:
total_rows = sum(len(df) for df in dfs.values())
total_usable = sum(
    int(df["usable_for_analysis"].sum())
    for df in dfs.values()
    if "usable_for_analysis" in df.columns
)
total_review = sum(
    int(df["needs_manual_review"].sum())
    for df in dfs.values()
    if "needs_manual_review" in df.columns
)

print("=" * 60)
print("DAY 1 NOTEBOOK 01 - COMPLETE")
print("=" * 60)
print(f"  Total cleaned rows loaded:    {total_rows}")
print(f"  Rows usable for analysis:     {total_usable}  ({100*total_usable//max(total_rows,1)}%)")
print(f"  Rows needing manual review:   {total_review}  ({100*total_review//max(total_rows,1)}%)")
print()
print("  Schemas loaded:")
for schema, df in dfs.items():
    print(f"    {schema:<30} {len(df):>3} rows")
print()
print("  Outputs written:")
for p in sorted(FIGURES.glob("eda_*_day1.png")):
    print(f"    {p.relative_to(PROJECT_ROOT)}")
manual_q = TABLES / "day1_manual_review_queue.csv"
if manual_q.exists():
    print(f"    {manual_q.relative_to(PROJECT_ROOT)}")
print()
print("  Next: Day 2 — text features, embeddings, claim divergence scoring")
print("  See docs/day1_evidence_audit.md for full audit and Day 2 recommendations.")
print("=" * 60)